# TamilScript OCR: Kaggle GPU Training & Model Upgrade
### Training State-of-the-Art Handwritten Tamil OCR on Kaggle Cloud GPU (T4 / P100 / A100)

This notebook provides a complete, reproducible training pipeline for **TamilScript OCR**:
1. Ingests ~103k real handwritten Tamil word samples from the benchmark **`c3rl/IIIT-INDIC-HW-WORDS-Tamil`** dataset.
2. Applies diacritic-safe geometric and photometric data augmentations.
3. Trains an 8.2M-parameter CNN-BiLSTM-CTC architecture with Automatic Mixed Precision (AMP) and Cosine Annealing.
4. Evaluates on a strictly held-out test split, measuring CER, WER, and exact match accuracy.
5. Exports a lightweight, production-compatible checkpoint (`tamil_crnn_best.pt`) with full metadata and checksums ready for Render deployment.

## 1. System, CUDA & Environment Verification

In [ ]:
import os
import sys
import platform
import torch
import numpy as np
import PIL

print('=' * 60)
print('ENVIRONMENT AUDIT')
print(f'Python Version: {sys.version}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'Device Capability: {torch.cuda.get_device_capability(0)}')
    print(f'Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: No GPU detected. Please go to Notebook Settings -> Accelerator -> GPU T4 x2 or P100.')
print('=' * 60)

## 2. Dependencies Installation

In [ ]:
!pip install -q datasets transformers accelerate opencv-python-headless pillow

## 3. Dataset Ingestion & Verification (`IIIT-INDIC-HW-WORDS-Tamil`)
The **IIIT-INDIC-HW-WORDS-Tamil** dataset contains ~103,000 real offline handwritten word images with ground-truth transcriptions in Tamil script.

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt

print('Downloading / Loading c3rl/IIIT-INDIC-HW-WORDS-Tamil from Hugging Face...')
dataset = load_dataset('c3rl/IIIT-INDIC-HW-WORDS-Tamil')
print(dataset)

train_split = dataset['train']
val_split = dataset['validation']
test_split = dataset['test']

print(f'Train samples: {len(train_split):,}')
print(f'Validation samples: {len(val_split):,}')
print(f'Test samples: {len(test_split):,}')

first_item = train_split[0]
print('\nSample record structure:')
for k, v in first_item.items():
    print(f' - {k}: {type(v)}')


In [ ]:
# Display representative handwritten samples
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    sample = train_split[i * 10]
    ax.imshow(sample['image'].convert('L'), cmap='gray')
    ax.set_title(sample['text'], fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Deterministic Tamil Vocabulary & CTC Utilities
We strictly match the production 87-token vocabulary (Blank=0, Vowels, Consonants, Grantha, Signs, Numerals, Punctuation) to ensure zero-downtime compatibility.

In [ ]:
import unicodedata

BLANK_TOKEN = '<blank>'
TAMIL_VOWELS = ['அ', 'ஆ', 'இ', 'ஈ', 'உ', 'ஊ', 'எ', 'ஏ', 'ஐ', 'ஒ', 'ஓ', 'ஔ', 'ஃ']
TAMIL_CONSONANTS = ['க', 'ங', 'ச', 'ஞ', 'ட', 'ண', 'த', 'ந', 'ப', 'ம', 'ய', 'ர', 'ல', 'வ', 'ழ', 'ள', 'ற', 'ன']
TAMIL_GRANTHA = ['ஜ', 'ஷ', 'ஸ', 'ஹ', 'க்ஷ', 'ஸ்ரீ']
TAMIL_SIGNS = ['்', 'ா', 'ி', 'ீ', 'ு', 'ூ', 'ெ', 'ே', 'ை', 'ொ', 'ோ', 'ௌ']
TAMIL_NUMERALS = ['௦', '௧', '௨', '௩', '௪', '௫', '௬', '௭', '௮', '௯', '௰', '௱', '௲']
ASCII_PUNCTUATION_AND_DIGITS = [
    ' ', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    '.', ',', '!', '?', '-', ':', ';', "'", '"', '(', ')', '/', '%'
]

VOCABULARY = [BLANK_TOKEN] + TAMIL_VOWELS + TAMIL_CONSONANTS + TAMIL_GRANTHA + TAMIL_SIGNS + TAMIL_NUMERALS + ASCII_PUNCTUATION_AND_DIGITS
CHAR_TO_IDX = {c: i for i, c in enumerate(VOCABULARY)}
IDX_TO_CHAR = {i: c for i, c in enumerate(VOCABULARY)}
VOCAB_SIZE = len(VOCABULARY)
print(f'Deterministic Vocabulary Size: {VOCAB_SIZE} tokens')

def normalize_text(text: str) -> str:
    if not text:
        return ''
    clean = text.replace('\r\n', ' ').replace('\r', ' ').replace('\n', ' ').strip()
    return unicodedata.normalize('NFC', clean)

def text_to_indices(text: str) -> list:
    norm = normalize_text(text)
    return [CHAR_TO_IDX[c] for c in norm if c in CHAR_TO_IDX]

def ctc_decode_greedy(predictions: list) -> str:
    collapsed = []
    prev_idx = -1
    for idx in predictions:
        if idx != prev_idx:
            if idx != 0 and idx in IDX_TO_CHAR:
                collapsed.append(IDX_TO_CHAR[idx])
            prev_idx = idx
    return normalize_text(''.join(collapsed))


## 5. Diacritic-Preserving Data Augmentation & PyTorch Dataset

In [ ]:
import random
import cv2
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader

class TamilDataset(Dataset):
    def __init__(self, hf_split, is_train: bool = True, target_height: int = 32, max_width: int = 768):
        self.dataset = hf_split
        self.is_train = is_train
        self.target_height = target_height
        self.max_width = max_width

    def __len__(self):
        return len(self.dataset)

    def augment(self, pil_img: Image.Image) -> Image.Image:
        img = pil_img.convert('L')
        if not self.is_train:
            return img
        if random.random() < 0.4:
            angle = random.uniform(-3.5, 3.5)
            img = img.rotate(angle, resample=Image.BILINEAR, expand=True, fillcolor=255)
        if random.random() < 0.35:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(random.uniform(0.85, 1.25))
        if random.random() < 0.25:
            arr = np.array(img)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
            arr = cv2.erode(arr, kernel, iterations=1) if random.random() > 0.5 else cv2.dilate(arr, kernel, iterations=1)
            img = Image.fromarray(arr)
        return img

    def preprocess(self, pil_img: Image.Image) -> torch.Tensor:
        aug_img = self.augment(pil_img)
        gray = np.array(aug_img.convert('L'))
        h, w = gray.shape
        if h == 0 or w == 0:
            return torch.ones((1, self.target_height, 64), dtype=torch.float32)
        scale = self.target_height / float(h)
        new_w = max(16, min(int(w * scale), self.max_width))
        resized = cv2.resize(gray, (new_w, self.target_height), interpolation=cv2.INTER_AREA)
        norm_img = resized.astype(np.float32) / 255.0
        if np.mean(norm_img) < 0.3:
            norm_img = 1.0 - norm_img
        return torch.tensor(norm_img, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        tensor_img = self.preprocess(row['image'])
        norm_text = normalize_text(row['text'])
        indices = text_to_indices(norm_text)
        return {
            'image': tensor_img,
            'text': norm_text,
            'target': torch.tensor(indices, dtype=torch.long),
            'width': tensor_img.shape[-1]
        }

def ctc_collate_fn(batch):
    batch_size = len(batch)
    max_w = max(item['width'] for item in batch)
    padded_images = torch.ones((batch_size, 1, 32, max_w), dtype=torch.float32)
    targets = []
    target_lengths = []
    input_lengths = []
    texts = []
    for i, item in enumerate(batch):
        img = item['image']
        w = item['width']
        padded_images[i, :, :, :w] = img
        targets.append(item['target'])
        target_lengths.append(len(item['target']))
        input_lengths.append(max(1, w // 4))
        texts.append(item['text'])
    targets_concat = torch.cat(targets, dim=0) if targets else torch.tensor([], dtype=torch.long)
    return {
        'images': padded_images,
        'targets': targets_concat,
        'input_lengths': torch.tensor(input_lengths, dtype=torch.long),
        'target_lengths': torch.tensor(target_lengths, dtype=torch.long),
        'texts': texts
    }


## 6. Model Architecture — Candidate A (`TamilHandwrittenCRNN`)
Exact 1:1 match with our production architecture (7-layer CNN + 2-layer BiLSTM + CTC).

In [ ]:
import torch.nn as nn

class TamilHandwrittenCRNN(nn.Module):
    def __init__(self, num_classes: int = VOCAB_SIZE, hidden_size: int = 256, in_channels: int = 1):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(512, 512, kernel_size=(2, 1), stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            dropout=0.25,
            batch_first=False
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        conv = self.cnn(x).squeeze(2).permute(2, 0, 1)
        rnn, _ = self.rnn(conv)
        logits = self.fc(rnn)
        return nn.functional.log_softmax(logits, dim=2)

model = TamilHandwrittenCRNN()
print(f'Candidate A Model Parameters: {sum(p.numel() for p in model.parameters()):,}')


## 7. Model Training Loop with AMP, Early Stopping & Checkpointing

In [ ]:
import time
import csv

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def eval_metrics(model, dataloader, device, max_samples=800):
    model.eval()
    char_edits, total_chars = 0, 0
    word_edits, total_words = 0, 0
    exact_matches, samples = 0, 0
    with torch.no_grad():
        for batch in dataloader:
            if samples >= max_samples:
                break
            imgs = batch['images'].to(device)
            log_probs = model(imgs)
            preds = torch.argmax(torch.exp(log_probs), dim=2).permute(1, 0)
            for i in range(imgs.size(0)):
                if samples >= max_samples:
                    break
                pred_text = ctc_decode_greedy(preds[i].cpu().numpy().tolist())
                gt_text = batch['texts'][i]
                edits = levenshtein(list(gt_text), list(pred_text))
                char_edits += edits
                total_chars += max(1, len(gt_text))
                w_edits = levenshtein(gt_text.split(), pred_text.split())
                word_edits += w_edits
                total_words += max(1, len(gt_text.split()))
                if gt_text == pred_text:
                    exact_matches += 1
                samples += 1
    return {
        'cer': char_edits / float(total_chars) if total_chars else 1.0,
        'wer': word_edits / float(total_words) if total_words else 1.0,
        'exact': exact_matches / float(samples) * 100.0 if samples else 0.0
    }


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TamilHandwrittenCRNN().to(device)

BATCH_SIZE = 64
EPOCHS = 25
LR = 5e-4

train_loader = DataLoader(TamilDataset(train_split, is_train=True), batch_size=BATCH_SIZE, shuffle=True, collate_fn=ctc_collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(TamilDataset(val_split, is_train=False), batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn, num_workers=2)
test_loader = DataLoader(TamilDataset(test_split, is_train=False), batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn, num_workers=2)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
best_cer = 1.0
patience, patience_limit = 0, 6

print(f'Commencing training for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_loss, batches = 0.0, 0
    for b in train_loader:
        imgs, tgts = b['images'].to(device), b['targets'].to(device)
        in_lens, tgt_lens = b['input_lengths'].to(device), b['target_lengths'].to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                loss = criterion(model(imgs), tgts, in_lens, tgt_lens)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(imgs), tgts, in_lens, tgt_lens)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        total_loss += loss.item()
        batches += 1
    
    scheduler.step()
    val_res = eval_metrics(model, val_loader, device)
    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Loss: {total_loss/batches:.4f} | Val CER: {val_res["cer"]*100:.2f}% | Val WER: {val_res["wer"]*100:.2f}% | Time: {elapsed:.1f}s')
    
    if val_res['cer'] < best_cer:
        best_cer = val_res['cer']
        patience = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_cer': best_cer, 'vocab_size': VOCAB_SIZE}, '/kaggle/working/checkpoints/tamil_crnn_best.pt')
        print(f' >> Saved best model with Val CER: {best_cer*100:.2f}%')
    else:
        patience += 1
        if patience >= patience_limit:
            print('Early stopping triggered!')
            break


## 8. Final Evaluation on Untouched Test Set

In [ ]:
# Load best checkpoint
best_ckpt = torch.load('/kaggle/working/checkpoints/tamil_crnn_best.pt', map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
test_results = eval_metrics(model, test_loader, device, max_samples=2500)

print('=' * 60)
print('FINAL TEST EVALUATION ON UNTOUCHED BENCHMARK SAMPLES:')
print(f" - Character Error Rate (CER): {test_results['cer'] * 100:.2f}%")
print(f" - Word Error Rate (WER):      {test_results['wer'] * 100:.2f}%")
print(f" - Exact Match Accuracy:      {test_results['exact']:.2f}%")
print('=' * 60)


## 9. Production Model Export & Checksum Verification
Exports clean inference weights (~31.4 MB, no optimizer states) directly to `/kaggle/working/tamil_crnn_best.pt`.

In [ ]:
import hashlib
import json

export_path = '/kaggle/working/tamil_crnn_best.pt'
clean_payload = {
    'model_state_dict': model.state_dict(),
    'vocab_size': VOCAB_SIZE,
    'val_cer': test_results['cer'],
    'architecture': 'TamilHandwrittenCRNN'
}
torch.save(clean_payload, export_path)

file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
sha256 = hashlib.sha256(open(export_path, 'rb').read()).hexdigest()
print(f'Exported: {export_path} ({file_size_mb:.2f} MB)')
print(f'SHA-256 Checksum: {sha256}')

with open('/kaggle/working/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'TamilScript OCR',
        'model_architecture': 'TamilHandwrittenCRNN',
        'sha256': sha256,
        'vocab_size': VOCAB_SIZE,
        'test_cer': test_results['cer'],
        'test_wer': test_results['wer'],
        'exact_match': test_results['exact'],
        'file_size_mb': round(file_size_mb, 2)
    }, f, indent=2)

with open('/kaggle/working/baseline_vs_candidate.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Metric', 'Baseline (Synthetic)', 'Trained Candidate', 'Absolute Improvement'])
    writer.writerow(['CER', '98.54%', f"{test_results['cer']*100:.2f}%", f"{(0.9854 - test_results['cer'])*100:.2f}%"])
    writer.writerow(['WER', '100.00%', f"{test_results['wer']*100:.2f}%", f"{(1.0 - test_results['wer'])*100:.2f}%"])
    writer.writerow(['Exact Match', '0.00%', f"{test_results['exact']:.2f}%", f"{test_results['exact']:.2f}%"])

print('\nAll export artifacts created successfully in /kaggle/working/.')
